# Chapter 04-07 · Pipelines and cross-validation without leaking

**Label:** Core  |  **Time:** ~60 minutes  |  **Difficulty:** moderate

**Prerequisites:** 04-05 for the four leaks, 04-06 for the four preprocessing decisions, 04-03 for the
selection premium.

**Position in the learning path:** module 04, chapter 7 of 8.

---

## Why this matters

The last two chapters ended with the same promise: **04-07 makes it automatic.**

04-05 showed that a fitted step outside the split can turn a coin flip into 79.5% accuracy. 04-06 made
four fitted decisions and kept saying "inside the fold". Both relied on you remembering to do the right
thing, every time, on every project.

This chapter replaces remembering with **structure**. A `Pipeline` is a single object that contains every
fitted step *and* the model, so when a cross-validator refits it, every step is refitted - imputer,
scaler, encoder, selector - on that fold's training rows only. **Correctness stops being a discipline and
becomes the default.**

It also introduces the honest way to evaluate a hyperparameter search - **nested cross-validation** - and
then does something more useful than recommending it: it measures what the search was actually costing.
The answer here is **nothing at all**, and the 0.024 gap that looks like a selection premium turns out to
be something else entirely. Finding that out takes four numbers.

## What you will be able to do

- Build a `Pipeline` and a `ColumnTransformer` for mixed numeric and categorical data
- Explain what `cross_val_score` does to a pipeline, step by step
- Measure the optimism of a manual workflow against a pipeline, on real data
- Use `GridSearchCV`, and say what its `best_score_` is and is not
- Set up nested cross-validation, and decide when it is worth its cost
- Ship one object that carries its own preprocessing

## Warm-up: retrieve, do not reread

1. Which preprocessing steps can leak the target?
2. What is the selection premium, and what makes it grow?
3. In 04-06, why was "drop incomplete rows scores 51.41" not evidence that dropping rows is better?

<br>

*Answers: (1) those that look at `y` - target encoding, feature selection. (2) the gap between a score on
the set that chose a model and a set that did not; it grows with the number of candidates tried. (3) it
was measured on different rows.*

## The data

**California housing** - the 1990 US census districts from 02-08, 20,640 rows, documented in
`data/README.md`. Two changes so that it exercises everything in this chapter:

- a **categorical** column, `region`, derived from latitude
- **missing values** punched into `MedInc` and `HouseAge`, so an imputer is needed

The target is the median house value in a district, in units of $100,000.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing

warnings.filterwarnings("ignore")

raw = fetch_california_housing(as_frame=True)
housing = raw.frame.copy()

# a categorical column, and holes, so the pipeline has real work to do
rng = np.random.default_rng(3)
housing["region"] = pd.cut(housing.Latitude, bins=[32, 34, 36, 38, 40, 43],
                           labels=["far south", "south", "central", "north", "far north"]).astype(str)
housing.loc[rng.random(len(housing)) < 0.10, "MedInc"] = np.nan
housing.loc[rng.random(len(housing)) < 0.08, "HouseAge"] = np.nan

NUMERIC = [c for c in raw.frame.columns if c != "MedHouseVal"]
CATEGORICAL = ["region"]

# a 4,000-row sample, so that everything in this chapter runs in seconds
sample = housing.sample(4000, random_state=0)
features = sample[NUMERIC + CATEGORICAL]
target = sample.MedHouseVal

print("%d districts, %d numeric columns, %d categorical" % (len(sample), len(NUMERIC), len(CATEGORICAL)))
print("missing:", {c: int(v) for c, v in sample[NUMERIC].isna().sum().items() if v})
print("regions:", sample.region.value_counts().to_dict())

## The two workflows, drawn

Here is the entire argument of this chapter in one picture. Same steps, same data; only the position of
the split differs.

In [ ]:
fig, (top, bottom) = plt.subplots(2, 1, figsize=(12.5, 6.2))

OFFSET = 1.7   # leave room on the left for the pipeline's wall and its label


def draw_flow(ax, title, boxes, split_after, verdict, verdict_colour):
    for position, (label, colour) in enumerate(boxes):
        left_edge = OFFSET + position * 2.05
        ax.add_patch(plt.Rectangle((left_edge, 0.3), 1.8, 0.9, facecolor=colour,
                                   edgecolor="white", linewidth=3))
        ax.text(left_edge + 0.9, 0.75, label, ha="center", va="center", fontsize=10)
        if position < len(boxes) - 1:
            ax.annotate("", xy=(left_edge + 2.0, 0.75), xytext=(left_edge + 1.85, 0.75),
                        arrowprops=dict(arrowstyle="-|>", color="#666666", linewidth=2))
    wall = OFFSET + split_after * 2.05 - 0.12
    ax.axvline(wall, color="#000000", linewidth=3)
    ax.text(wall, 1.44, "the split happens here", ha="center", fontsize=10, fontweight="bold")
    ax.text(OFFSET + len(boxes) * 2.05 - 0.15, 0.75, verdict, ha="left", va="center", fontsize=11,
            color=verdict_colour, fontweight="bold")
    ax.set_title(title, fontsize=12, loc="left")
    ax.set_xlim(-0.15, OFFSET + len(boxes) * 2.05 + 3.4)
    ax.set_ylim(0, 1.85)
    ax.set_xticks([]); ax.set_yticks([])
    for side in ax.spines.values():
        side.set_visible(False)

steps = [("impute", "#cfe3f3"), ("scale", "#cfe3f3"), ("encode", "#cfe3f3"),
         ("select", "#f6d3bd"), ("fit model", "#cfe8dc")]
draw_flow(top, "The manual workflow: prepare everything, then split", steps, 4,
          "leaks", "#D55E00")
draw_flow(bottom, "The pipeline: split first, then prepare inside each fold", steps, 0,
          "honest", "#009E73")
plt.tight_layout()
plt.show()

**In the top row, four fitted steps have already seen every row before the split exists.** In the bottom
row the split comes first and everything after it is refitted on training rows only.

The steps are identical. **The only difference is where the black line sits**, and that is exactly what a
`Pipeline` controls - because it turns the four boxes plus the model into a *single estimator* that the
cross-validator can refit as a unit.

## Building it

Two objects do the work.

**`ColumnTransformer`** routes columns to different treatments - numeric columns get imputed and scaled,
the categorical column gets one-hot encoded - and glues the results back together.

**`Pipeline`** chains steps so the whole thing behaves as one estimator with one `fit` and one `predict`.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocess = ColumnTransformer([
    ("numeric", Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), NUMERIC),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
])

model = Pipeline([("prepare", preprocess), ("estimate", Ridge())])
print(model)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.6))

ax.add_patch(plt.Rectangle((0.1, 0.4), 1.9, 3.4, facecolor="#eef4f9", edgecolor="#0072B2",
                           linewidth=2))
ax.text(1.05, 3.55, "one DataFrame", ha="center", fontsize=10, fontweight="bold")
for position, name in enumerate(NUMERIC + CATEGORICAL):
    ax.add_patch(plt.Rectangle((0.25, 3.05 - position * 0.33), 1.6, 0.26,
                               facecolor="#f6d3bd" if name == "region" else "#cfe3f3",
                               edgecolor="white", linewidth=1.5))
    ax.text(1.05, 3.18 - position * 0.33, name, ha="center", va="center", fontsize=8.5)

ax.annotate("", xy=(3.0, 2.7), xytext=(2.05, 2.4),
            arrowprops=dict(arrowstyle="-|>", color="#0072B2", linewidth=2.2))
ax.annotate("", xy=(3.0, 1.1), xytext=(2.05, 1.2),
            arrowprops=dict(arrowstyle="-|>", color="#D55E00", linewidth=2.2))

ax.add_patch(plt.Rectangle((3.05, 2.15), 3.5, 1.1, facecolor="#cfe3f3", edgecolor="white", linewidth=3))
ax.text(4.8, 2.92, "8 numeric columns", ha="center", fontsize=10, fontweight="bold")
ax.text(4.8, 2.48, "SimpleImputer(median)\nthen StandardScaler", ha="center", fontsize=9,
        color="#444444")

ax.add_patch(plt.Rectangle((3.05, 0.6), 3.5, 1.1, facecolor="#f6d3bd", edgecolor="white", linewidth=3))
ax.text(4.8, 1.37, "1 categorical column", ha="center", fontsize=10, fontweight="bold")
ax.text(4.8, 0.95, "OneHotEncoder\nhandle_unknown='ignore'", ha="center", fontsize=9, color="#444444")

ax.annotate("", xy=(7.5, 1.9), xytext=(6.6, 2.7),
            arrowprops=dict(arrowstyle="-|>", color="#0072B2", linewidth=2.2))
ax.annotate("", xy=(7.5, 1.9), xytext=(6.6, 1.15),
            arrowprops=dict(arrowstyle="-|>", color="#D55E00", linewidth=2.2))
ax.add_patch(plt.Rectangle((7.55, 1.35), 2.5, 1.1, facecolor="#cfe8dc", edgecolor="white", linewidth=3))
ax.text(8.8, 2.12, "one matrix", ha="center", fontsize=10, fontweight="bold")
ax.text(8.8, 1.70, "8 + 5 = 13 columns", ha="center", fontsize=9, color="#444444")

ax.annotate("", xy=(11.0, 1.9), xytext=(10.1, 1.9),
            arrowprops=dict(arrowstyle="-|>", color="#666666", linewidth=2.2))
ax.add_patch(plt.Rectangle((11.05, 1.5), 1.5, 0.8, facecolor="#999999", edgecolor="white", linewidth=3))
ax.text(11.8, 1.9, "Ridge", ha="center", va="center", fontsize=11, color="white", fontweight="bold")

ax.set_xlim(0, 12.9)
ax.set_ylim(0.2, 4.0)
ax.set_xticks([]); ax.set_yticks([])
for side in ax.spines.values():
    side.set_visible(False)
ax.set_title("ColumnTransformer routes columns; Pipeline chains the result into the model",
             fontsize=12)
plt.tight_layout()
plt.show()

### What `cross_val_score` does to that object

This is worth spelling out, because the whole safety property lives here.

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

folds = KFold(5, shuffle=True, random_state=0)

fig, ax = plt.subplots(figsize=(11.5, 4.0))
for fold, (train_rows, test_rows) in enumerate(folds.split(features)):
    for start, width, colour in [(0, 1, "#0072B2")]:
        pass
    marks = np.zeros(len(features))
    marks[test_rows] = 1
    chunk = len(features) // 5
    for block in range(5):
        is_test = block == fold
        ax.add_patch(plt.Rectangle((block * 2.0, -fold - 0.35), 1.9, 0.7,
                                   facecolor="#D55E00" if is_test else "#0072B2",
                                   edgecolor="white", linewidth=2))
        ax.text(block * 2.0 + 0.95, -fold, "test" if is_test else "train", ha="center", va="center",
                color="white", fontsize=9)
    ax.text(10.3, -fold, "fit the WHOLE pipeline on the blue rows,\nscore it on the orange ones",
            va="center", fontsize=8.5, color="#444444")
ax.set_xlim(-0.2, 16.4)
ax.set_ylim(-4.8, 0.9)
ax.set_yticks([-f for f in range(5)])
ax.set_yticklabels(["fold %d" % (f + 1) for f in range(5)], fontsize=9)
ax.set_xticks([])
for side in ax.spines.values():
    side.set_visible(False)
ax.set_title("cross_val_score refits every step of the pipeline, five times", fontsize=12)
plt.tight_layout()
plt.show()

scores = cross_val_score(model, features, target, cv=folds, scoring="neg_mean_absolute_error")
print("ridge in a pipeline: MAE %.4f  (folds %s)" % (-scores.mean(), np.round(-scores, 3)))
print("fold-to-fold sd %.4f - California districts differ a lot from each other" % scores.std())

**The imputer's median, the scaler's mean and standard deviation, and the encoder's list of regions are
all recomputed five times**, each from four-fifths of the data. No test row contributed to any of them.

That is the entire mechanism, and it is why `Pipeline` is not a convenience wrapper. **A loose
`StandardScaler` fitted before `cross_val_score` cannot be refitted by it**, because the cross-validator
never sees it.

## Measuring what the pipeline prevents

04-05 measured this on pure noise, where selection leakage was worth 0.25 of accuracy. **How much is it
worth on real data with a real signal?**

**Predict before running.**

In [ ]:
from sklearn.base import clone
from sklearn.feature_selection import SelectKBest, f_regression

rows = []
for n_rows in [400, 4000]:
    part = housing.sample(n_rows, random_state=1)
    part_features, part_target = part[NUMERIC + CATEGORICAL], part.MedHouseVal

    # the manual workflow: prepare and select using every row, then cross-validate the model alone
    prepared = clone(preprocess).fit_transform(part_features, part_target)
    selected = SelectKBest(f_regression, k=3).fit(prepared, part_target).transform(prepared)
    manual = -cross_val_score(Ridge(), selected, part_target, cv=folds,
                              scoring="neg_mean_absolute_error").mean()

    # the pipeline: identical steps, all inside
    inside = Pipeline([("prepare", clone(preprocess)),
                       ("select", SelectKBest(f_regression, k=3)),
                       ("estimate", Ridge())])
    honest = -cross_val_score(inside, part_features, part_target, cv=folds,
                              scoring="neg_mean_absolute_error").mean()

    rows.append({"rows": n_rows, "manual workflow": round(manual, 4),
                 "pipeline": round(honest, 4), "optimism": round(honest - manual, 4)})
leak_table = pd.DataFrame(rows)
print(leak_table.to_string(index=False))

**0.0181 on 400 rows and 0.0109 on 4,000** - real, consistently in the same direction, and far smaller
than 04-05's 0.25.

That contrast is the useful part, and it has a clean explanation from 04-05's severity principle. **The
size of a selection leak depends on how much of what the selector found was luck.** On pure noise,
*everything* it found was luck. Here the strongest columns are genuinely strongest - median income really
does predict house value - so the selector picks much the same three columns whether or not it saw the
test rows, and the leak is confined to the marginal cases.

Two things follow, and they pull in opposite directions on purpose:

- **You cannot tell which regime you are in without checking.** The same code produced 0.25 and 0.01. The
  difference was the data, not the mistake.
- **The pipeline costs nothing**, so this measurement is not a licence to skip it. It tells you how
  urgently to fix code that already exists, which is a different question from how to write new code.

## Searching for hyperparameters

`GridSearchCV` wraps a pipeline, tries every combination of settings you list, cross-validates each, and
keeps the best. Because the pipeline is the thing being refitted, **the preprocessing is redone inside
every fold of every candidate** - which is what makes a search over preprocessing choices legitimate.

Note the `step__parameter` naming: `estimate__alpha` reaches into the step called `estimate`.

In [ ]:
from sklearn.model_selection import GridSearchCV

search = GridSearchCV(
    model,
    {"estimate__alpha": [0.001, 0.01, 0.1, 1, 10, 100, 1000],
     "prepare__numeric__impute__strategy": ["mean", "median"]},
    cv=folds, scoring="neg_mean_absolute_error")
search.fit(features, target)

results = pd.DataFrame(search.cv_results_)
print("tried %d combinations" % len(results))
print("best: alpha=%s, imputer=%s"
      % (search.best_params_["estimate__alpha"],
         search.best_params_["prepare__numeric__impute__strategy"]))
print("its cross-validated MAE: %.4f" % -search.best_score_)

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.5))
for strategy, colour in [("mean", "#0072B2"), ("median", "#D55E00")]:
    subset = results[results["param_prepare__numeric__impute__strategy"] == strategy]
    subset = subset.sort_values("param_estimate__alpha")
    ax.errorbar(subset["param_estimate__alpha"].astype(float),
                -subset.mean_test_score, yerr=subset.std_test_score,
                fmt="o-", capsize=4, color=colour, label="imputer: %s" % strategy)
ax.axvline(search.best_params_["estimate__alpha"], color="#000000", linestyle="--", linewidth=1.2)
ax.text(search.best_params_["estimate__alpha"] * 1.3, ax.get_ylim()[1] * 0.98,
        "chosen", fontsize=9, va="top")
ax.set_xscale("log")
ax.set_xlabel("ridge alpha (log scale)")
ax.set_ylabel("cross-validated MAE")
ax.set_title("Fourteen candidates, and error bars that overlap almost everywhere", fontsize=11.5)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Look at the error bars before looking at the winner.** The fold-to-fold spread is far larger than the
differences between most candidates - the choice between `mean` and `median` imputation is invisible
inside it, and so is every alpha below 100.

That is the normal situation, and it is worth internalising: **most hyperparameter searches are choosing
between options that the data cannot distinguish.** The search still returns a winner, confidently, and
`best_score_` is the score of that winner **on the folds that chose it**.

Which brings us to what that number is not.

## `best_score_` is not an estimate of performance

It is the maximum of fourteen noisy numbers, and 04-03 measured what taking a maximum does. The honest
version is **nested cross-validation**: an inner loop chooses the hyperparameters, an outer loop scores
the *whole procedure including the choosing*.

In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 4.4))
for outer_fold in range(4):
    for block in range(4):
        is_outer_test = block == outer_fold
        ax.add_patch(plt.Rectangle((block * 1.6, -outer_fold * 1.25 - 0.3), 1.5, 0.6,
                                   facecolor="#D55E00" if is_outer_test else "#cfe3f3",
                                   edgecolor="white", linewidth=2))
        if is_outer_test:
            ax.text(block * 1.6 + 0.75, -outer_fold * 1.25, "score", ha="center", va="center",
                    color="white", fontsize=8.5)
    for inner in range(3):
        ax.add_patch(plt.Rectangle((6.8 + inner * 1.15, -outer_fold * 1.25 - 0.3), 1.05, 0.6,
                                   facecolor="#0072B2" if inner < 2 else "#e8a33d",
                                   edgecolor="white", linewidth=2))
    ax.text(10.5, -outer_fold * 1.25, "inner loop picks alpha\nusing only the blue rows",
            va="center", fontsize=8.5, color="#444444")
ax.text(2.4, 0.7, "OUTER: hold this fold back entirely", ha="center", fontsize=10, fontweight="bold")
ax.text(8.5, 0.7, "INNER: search, inside the rest", ha="center", fontsize=10, fontweight="bold")
ax.set_xlim(-0.2, 15.6)
ax.set_ylim(-4.4, 1.2)
ax.set_xticks([]); ax.set_yticks([])
for side in ax.spines.values():
    side.set_visible(False)
ax.set_title("Nested cross-validation: the search happens inside each outer training set", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
inner_folds = KFold(4, shuffle=True, random_state=1)
grid = {"estimate__alpha": [0.001, 0.01, 0.1, 1, 10, 100, 1000],
        "prepare__numeric__impute__strategy": ["mean", "median"]}

started = time.perf_counter()
one_search = GridSearchCV(model, grid, cv=inner_folds, scoring="neg_mean_absolute_error")
one_search.fit(features, target)
single_seconds = time.perf_counter() - started

started = time.perf_counter()
nested = -cross_val_score(GridSearchCV(model, grid, cv=inner_folds,
                                       scoring="neg_mean_absolute_error"),
                          features, target, cv=folds, scoring="neg_mean_absolute_error")
nested_seconds = time.perf_counter() - started

print("%-46s %8s %10s" % ("", "MAE", "seconds"))
print("%-46s %8.4f %10.1f" % ("best_score_ from one search", -one_search.best_score_, single_seconds))
print("%-46s %8.4f %10.1f" % ("nested cross-validation", nested.mean(), nested_seconds))
print()
print("the gap  : %+.4f" % (nested.mean() - (-one_search.best_score_)))
print("the price: %.1f times the compute" % (nested_seconds / single_seconds))

**A gap of 0.024, at roughly five times the compute** (the exact ratio moves with the machine). The
obvious reading is that `best_score_` was optimistic by
0.024 because it is the maximum of fourteen noisy numbers - 04-03's selection premium, arriving on
schedule.

**That reading is wrong, and this data can prove it.** The two procedures differ in *two* ways at once,
not one: nested cross-validation searches, and its inner loop also trains on less data. Separate them.

In [ ]:
chosen = one_search.best_params_


def plain(cross_validator, settings=None):
    candidate = Pipeline([("prepare", preprocess), ("estimate", Ridge())])
    if settings:
        candidate.set_params(**settings)
    return -cross_val_score(candidate, features, target, cv=cross_validator,
                            scoring="neg_mean_absolute_error").mean()


rows = [
    {"what": "A  best_score_ (inner 4-fold, trains on 3/4)", "MAE": round(-one_search.best_score_, 4)},
    {"what": "B  nested (outer 5-fold, inner trains on 3/5)", "MAE": round(nested.mean(), 4)},
    {"what": "C  plain 5-fold, chosen settings, no search", "MAE": round(plain(folds, chosen), 4)},
    {"what": "D  plain 4-fold, chosen settings, no search", "MAE": round(plain(inner_folds, chosen), 4)},
]
print(pd.DataFrame(rows).to_string(index=False))
print()
print("B - C  the selection premium (search vs no search, same folds) : %+.4f"
      % (nested.mean() - plain(folds, chosen)))
print("C - D  the training-size effect (4/5 of the rows vs 3/4)       : %+.4f"
      % (plain(folds, chosen) - plain(inner_folds, chosen)))

**The selection premium is +0.0000, and the whole 0.024 is the fold count.**

Nested cross-validation returns *exactly* what plain 5-fold cross-validation of the chosen settings
returns. Searching fourteen candidates cost nothing at all, and `best_score_` looked lower only because a
4-fold inner loop trains each model on 3/4 of the rows instead of 4/5 - less data, worse model, better-
looking error.

This is the third time in module 04 that a comparison has changed two things at once. **04-04's stricter
split was also a smaller one. 04-06's row-dropping was also an easier problem. And here, the honest
procedure also trains on less data.** It is worth stating as a habit:

> **Before attributing a difference to the thing you were investigating, list everything else that
> changed.** Then hold those fixed and measure again.

### So when does the premium actually bite?

04-03 measured a premium of 0.12 from forty candidates on 104 rows. Here fourteen candidates on 4,000
rows give zero. Two things differ, and both matter:

- **Candidates relative to data.** Forty choices over a hundred rows is a search with room to exploit
  noise. Fourteen over four thousand is not.
- **How *different* the candidates are.** Ridge with alpha 0.001 and alpha 0.01 are very nearly the same
  model, so fourteen alphas are nowhere near fourteen independent chances to get lucky. 04-03's candidates
  were random four-column subsets of a table containing twenty noise columns - genuinely different, and
  mostly worthless. **The premium counts *effective* candidates, not entries in a grid.**

So the practical rule is not "always nest", and it is not "never bother":

| Situation | What to do |
|---|---|
| Few, similar candidates; plenty of data | the premium is likely negligible - **check it once**, as above, then report `best_score_` with the candidate count |
| Many genuinely different candidates, or little data | **nest**, or keep an untouched test set |
| You already hold out a test set | that is the honest number, and it costs nothing extra |
| Anybody will act on the result | nest or hold out. `best_score_` alone is never a result |

**The cheapest correct option on a decent-sized dataset remains 04-03's:** one test set, split off at the
start, never looked at until the end. Nested cross-validation is what you use when the data is too small
to give a fifth of it away - which is exactly the regime where the premium is largest, so the two
recommendations agree rather than compete.

## One object, shipped

The last argument for pipelines has nothing to do with leakage.

In [ ]:
import pickle

fitted = search.best_estimator_.fit(features, target)

# a single listing arrives, with a hole in it and a region the encoder has seen
arriving = features.iloc[[0]].copy()
arriving.loc[:, "MedInc"] = np.nan

print("one object handles all of it:")
print("  input has a missing MedInc and a text region -> prediction %.4f"
      % fitted.predict(arriving)[0])
print()
print("  it carries %d fitted steps:" % len(fitted.named_steps))
for name, step in fitted.named_steps.items():
    print("    %-10s %s" % (name, type(step).__name__))
print()
print("  serialised size: %.1f KB" % (len(pickle.dumps(fitted)) / 1024))

**The imputer's medians, the scaler's statistics, the encoder's region list and the fitted coefficients
travel together.** Deployment is one `pickle.load` and one `predict`, on a raw DataFrame with the same
column names.

The alternative - a loose scaler, a loose imputer, a loose encoder and a model, applied in the right
order from memory - is where a large share of production bugs live. 04-06's E16 named two of them: an
unseen category changing the column count, and columns arriving in a different order. **A
`ColumnTransformer` addressing columns by name solves both, and it solves them in the object you shipped
rather than in a runbook.**

## The whole module, in one figure

In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 5.6))
stages = [
    ("04-01  Frame it", "#cfe3f3", "unit, target, prediction time, horizon, availability"),
    ("04-02  Baseline it", "#cfe3f3", "compute what free rules score, before any model"),
    ("04-03  Split it", "#f6d3bd", "train / validation / test; the test set is read once"),
    ("04-04  Split it properly", "#f6d3bd", "group by entity, and split by time, when they apply"),
    ("04-05  Check for leaks", "#f3d6e3", "four kinds; ask what each step learned about y"),
    ("04-06  Preprocess", "#cfe8dc", "impute, encode, scale, transform - all fitted steps"),
    ("04-07  Put it in a pipeline", "#cfe8dc", "one object; every step refitted inside every fold"),
]
for position, (title, colour, detail) in enumerate(stages):
    y = len(stages) - position - 1
    ax.add_patch(plt.Rectangle((0.05, y + 0.08), 2.9, 0.84, facecolor=colour, edgecolor="white",
                               linewidth=2.5))
    ax.text(1.5, y + 0.5, title, ha="center", va="center", fontsize=11, fontweight="bold")
    ax.text(3.1, y + 0.5, detail, va="center", fontsize=9.5, color="#333333")
    if position < len(stages) - 1:
        ax.annotate("", xy=(1.5, y + 0.02), xytext=(1.5, y + 0.08),
                    arrowprops=dict(arrowstyle="-|>", color="#888888", linewidth=1.6))
ax.set_xlim(0, 11.4)
ax.set_ylim(-0.15, len(stages) + 0.3)
ax.set_xticks([]); ax.set_yticks([])
for side in ax.spines.values():
    side.set_visible(False)
ax.set_title("The workflow module 04 has been building, in order", fontsize=13)
plt.tight_layout()
plt.show()

## Common misconceptions

**"A pipeline is a convenience wrapper."**
It is the object the cross-validator refits. A loose step outside it is never refitted, and therefore
never validated.

**"Cross-validation makes a result honest."**
It repeats the procedure you hand it. 04-05's coin-flip demonstration was five-fold cross-validated.

**"`GridSearchCV.best_score_` is my model's performance."**
It is the maximum of many noisy scores, measured on the folds that produced it. Whether that makes it
optimistic is a question you have to measure - here the selection premium was exactly zero.

**"Nested cross-validation is always required."**
It is one of two honest options, and it costs several times the compute for a correction that was zero on
this data. A held-out test set is the other, and it is cheaper when you can afford the rows.

**"Preprocessing leakage was the reason for all this."**
The measured cost of the leak here was 0.011 to 0.018. The reasons pipelines win are that they make
correctness automatic, they survive a step being added later, and they deploy as one object.

**"I'll build the pipeline once the experiments are done."**
The experiments are where the choices are made, and choices made on leaky evidence do not become correct
afterwards.

## Exercises

Solutions: `solutions/04_workflow/04-07_pipelines_cv_solutions.ipynb`.

### Quick understanding

**E1.** What does `cross_val_score` refit when given a `Pipeline`, and what does it refit when given a
bare model with preprocessing already applied?

**E2.** In `GridSearchCV`, what does the parameter name `prepare__numeric__impute__strategy` address?

**E3.** State in one sentence what `best_score_` measures, and one sentence on what it does not. Then
say what the 0.024 gap in this chapter turned out to be.

### Hand calculation

**E4.** A grid has 3 alphas, 2 imputers and 4 selector sizes, evaluated with 5-fold cross-validation.
Compute the number of model fits. Then compute it again for nested cross-validation with a 4-fold inner
loop and a 5-fold outer loop.

**E5.** A single fit takes 0.8 seconds. Using E4's counts, how long does the plain search take, and the
nested version? State the ratio and compare it with this chapter's measured 5x.

**E6.** Fourteen candidates are scored, and each score has a standard error of 0.05. Using 04-03's E6
arithmetic (the expected maximum of `k` standard normal draws is about 1.7 for k=14), estimate the
optimism of `best_score_`. Compare with the 0.03 measured here and comment on the difference.

**E7.** A `ColumnTransformer` receives 8 numeric columns and one categorical column with 5 levels, one-hot
encoded. How many columns does the model see? What if `drop="first"` is used, and why might you want that
for a linear model?

### Coding

**E8.** Build the pipeline from this chapter and confirm you can `fit` it on a DataFrame containing
missing values and strings, with no manual preparation at all.

**E9.** Add a `TransformedTargetRegressor` so the model predicts `log(MedHouseVal)` and back-transforms
automatically. Does it help here? Relate your answer to 04-06's finding about the back-transform.

**E10.** Replace `GridSearchCV` with `RandomizedSearchCV` over a wider space, with the same compute
budget. Which finds the better model, and what does that suggest about grids?

**E11.** Write `compare_pipelines(named_pipelines, X, y)` that cross-validates several pipelines on the
same folds and reports mean, spread and the paired difference from the best - the paired comparison from
04-03's E8. Use it to compare ridge, a random forest and gradient boosting.

**E12.** Use `cross_validate` instead of `cross_val_score` to return both training and test scores in one
call, and plot the gap by model. Which model overfits most, and does the ranking match the test scores?

### Interpretation

**E13.** A colleague reports "0.31 MAE, five-fold cross-validated, with hyperparameters tuned by grid
search". Name what is missing from that sentence and what you would ask for.

**E14.** Your nested cross-validation gives 0.69 and your single held-out test set gives 0.62. Which do
you report, and what explains the difference?

### Debugging

**E15.** `GridSearchCV` raises `Invalid parameter alpha for estimator Pipeline`. What is wrong and what is
the fix?

**E16.** A pipeline scores well but is enormous when pickled - hundreds of megabytes. Name the likely
step and the trade-off it represents.

### Exam and interview reasoning

**E17.** "Why do you use a pipeline?" Answer in under a minute with three reasons, only one of which is
leakage. Then: "when would you not bother?"

### Transfer to a different situation

**E18.** You have text (a product description), numbers (price, weight) and categories (brand, category)
predicting whether an item is returned. Sketch the `ColumnTransformer`, name the step most likely to leak,
and say which hyperparameter you would search first.

### Explain it to someone non-technical

**E19.** Explain in under 90 words why the model has to be "packaged with its own preparation
instructions", using an analogy.

### Optional challenge

**E20.** Measure how the optimism of `best_score_` grows with the size of the grid. For grids of 2, 6, 14,
30 and 60 combinations, compute `best_score_` and the nested score, and plot the gap. Compare the shape
with 04-03's E20 premium curve, and say whether the two agree.

In [ ]:
# Your workspace. In memory: housing, sample, features, target, NUMERIC, CATEGORICAL,
# preprocess, model, folds, inner_folds, grid, search, nested, leak_table.

## Mastery check

- [ ] Build a `ColumnTransformer` routing numeric and categorical columns differently
- [ ] Explain what `cross_val_score` refits, and why a loose step is never validated
- [ ] Measure the optimism of a manual workflow against a pipeline
- [ ] Address a nested parameter with `step__sub__parameter`
- [ ] Say what `best_score_` is and is not, and decompose a gap between it and a nested score
- [ ] Set up nested cross-validation and state its cost
- [ ] Ship a fitted pipeline and predict on a raw DataFrame with holes in it

## What should now feel instinctive

- Putting every fitted step inside the pipeline as the first draft, not a later tidy-up
- Reading `best_score_` as "the winner's score on the folds that chose it"
- Looking at the error bars on a hyperparameter sweep before the winner
- Asking "how many candidates?" whenever anybody quotes a tuned score
- Serialising the pipeline rather than the model

## Flashcards

| Front | Back |
|---|---|
| `Pipeline` | Chains fitted steps and a model into one estimator with one `fit` |
| `ColumnTransformer` | Routes columns to different treatments, then glues the results together |
| What CV refits | Every step of the pipeline, on every fold. Loose steps are never refitted |
| Parameter naming | `step__substep__parameter`, e.g. `prepare__numeric__impute__strategy` |
| Manual-workflow optimism here | +0.0181 on 400 rows, +0.0109 on 4,000 - against 0.25 on pure noise in 04-05 |
| `best_score_` | The maximum of many noisy scores, on the folds that produced it |
| Its selection premium here | **+0.0000** - fourteen similar candidates on 4,000 rows exploited nothing |
| Nested cross-validation | Inner loop chooses, outer loop scores the whole procedure |
| The 0.024 gap here | Not selection - the inner loop trains on 3/4 of the rows instead of 4/5 |
| What the premium counts | *Effective* candidates: fourteen nearby ridge alphas are not fourteen chances |
| What nesting costs | Several times the compute, and it grows with the grid |
| The cheaper honest option | One untouched test set, when you can afford the rows |
| Why ship the pipeline | The imputer, scaler, encoder and model travel together, addressed by column name |

## Next

**04-08 · Reproducibility, seeds, and trustworthy experiment records.** Module 04 has built a workflow
that is correct. The last chapter makes it *repeatable*: what a seed does and does not fix, why a pinned
environment matters more than people expect, and what an experiment record has to contain for somebody
else - or you in six months - to get your number back.

Then module 05 starts fitting models in earnest, on top of all of this.